[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/hamilton-certified/notebooks/day-09-async-parallel.ipynb#scrollTo=hh000001)

---
# Day 9 · Async & Parallel Execution
**certified-journeys / hamilton-certified** · Day 9 · Execution & Performance

> **Goal for today:** Rewrite a synchronous pipeline as async, fan out across multiple inputs with the parallel executor, and benchmark sequential vs. parallel execution.

In [ ]:
%pip install -q sf-hamilton

## When to Use Async vs Parallel

| Mode | Use when | Hamilton tool |
|---|---|---|
| Synchronous | CPU-bound transforms, pandas operations | `driver.Builder()` |
| **Async** | I/O-bound nodes (API calls, DB queries, file reads) | `async_driver.AsyncDriver` |
| **Parallel** | Same pipeline on many independent inputs | `h_threadpool` or `h_ray` adapter |

**Key rule:** async Hamilton is not faster for CPU-bound work (pandas, numpy). It shines when nodes wait for I/O — network calls, database queries, reading remote files. Python's GIL means asyncio still runs one node at a time CPU-wise, but the event loop can schedule other nodes during I/O waits.

The parallel executor is for a different shape of problem: same pipeline applied to N independent rows/entities.

In [ ]:
import sys, types, time, asyncio
import numpy as np
import pandas as pd
from hamilton import driver
from hamilton.async_driver import AsyncDriver
from hamilton.function_modifiers import tag

# Simulate I/O-bound node: sleep represents a DB query or API call
# Production equivalent: replace asyncio.sleep with an async DB client call

async def user_profile(user_id: int) -> dict:
    """Fetch user profile — simulates a 100ms async DB query."""
    await asyncio.sleep(0.1)  # simulated I/O wait
    return {'user_id': user_id, 'age': 20 + (user_id % 50), 'tier': user_id % 3}

async def purchase_history(user_id: int) -> dict:
    """Fetch purchase history — simulates a 150ms async API call."""
    await asyncio.sleep(0.15)  # different latency
    return {'total_spend': user_id * 12.5, 'order_count': user_id % 20 + 1}

@tag(feature_type='numerical')
async def spend_per_order(purchase_history: dict) -> float:
    """Derived: average spend per order."""
    return purchase_history['total_spend'] / max(purchase_history['order_count'], 1)

@tag(feature_type='numerical')
async def customer_age(user_profile: dict) -> int:
    """Derived: customer age from profile."""
    return user_profile['age']

@tag(feature_type='boolean')
async def is_premium(user_profile: dict, spend_per_order: float) -> bool:
    """Premium: tier 0 AND high spend per order."""
    return user_profile['tier'] == 0 and spend_per_order > 50.0

async_module = types.ModuleType('async_pipeline')
for fn in [user_profile, purchase_history, spend_per_order, customer_age, is_premium]:
    setattr(async_module, fn.__name__, fn)
sys.modules['async_pipeline'] = async_module

print('Async pipeline defined: user_profile + purchase_history → spend_per_order, customer_age, is_premium')

## Step 1 · AsyncDriver — Drop-in Replacement for Driver

`AsyncDriver` has the same Builder interface as the synchronous `Driver`. The only differences:
1. Import from `hamilton.async_driver`
2. Call `await dr.execute(...)` instead of `dr.execute(...)`
3. Your Hamilton functions can be `async def`

Non-async functions work too — `AsyncDriver` runs them synchronously.

In [ ]:
# Build the async driver — same Builder API
dr_async = (
    AsyncDriver.Builder()
    .with_modules(async_module)
    .build()
)

async def run_single():
    start = time.perf_counter()
    result = await dr_async.execute(
        ['spend_per_order', 'customer_age', 'is_premium'],
        inputs={'user_id': 42}
    )
    elapsed = time.perf_counter() - start
    return result, elapsed

result, elapsed = await run_single()
print(f'Result: {result}')
print(f'Elapsed: {elapsed:.3f}s  (nodes ran concurrently: user_profile + purchase_history in parallel)')

# Note: user_profile (0.1s) and purchase_history (0.15s) have no dependency between them
# AsyncDriver runs them concurrently — total ≈ 0.15s, not 0.1+0.15=0.25s
assert elapsed < 0.22, f'Expected < 0.22s (concurrent I/O), got {elapsed:.3f}s'

### What just happened?
- **`user_profile` and `purchase_history` ran concurrently** — they have no dependency between them, so AsyncDriver runs both at the same time.
- Total time ≈ 0.15s (the slower of the two), not 0.25s (sequential sum).
- **`spend_per_order` and `customer_age`** each waited for their respective dependency before running.
- The concurrency is automatic — Hamilton infers it from the DAG topology.

## Step 2 · Sequential vs Async — Timing Comparison

In [ ]:
# Synchronous version of the same pipeline (blocking sleeps)
import time as _time

def user_profile_sync(user_id: int) -> dict:
    _time.sleep(0.1)
    return {'user_id': user_id, 'age': 20 + (user_id % 50), 'tier': user_id % 3}

def purchase_history_sync(user_id: int) -> dict:
    _time.sleep(0.15)
    return {'total_spend': user_id * 12.5, 'order_count': user_id % 20 + 1}

def spend_per_order_sync(purchase_history_sync: dict) -> float:
    return purchase_history_sync['total_spend'] / max(purchase_history_sync['order_count'], 1)

def customer_age_sync(user_profile_sync: dict) -> int:
    return user_profile_sync['age']

sync_module = types.ModuleType('sync_pipeline')
for fn in [user_profile_sync, purchase_history_sync, spend_per_order_sync, customer_age_sync]:
    setattr(sync_module, fn.__name__, fn)
sys.modules['sync_pipeline'] = sync_module

dr_sync = driver.Builder().with_modules(sync_module).build()

# Benchmark both
t0 = time.perf_counter()
_ = dr_sync.execute(['spend_per_order_sync', 'customer_age_sync'], inputs={'user_id': 42})
sync_time = time.perf_counter() - t0

async def bench_async():
    t0 = time.perf_counter()
    await dr_async.execute(['spend_per_order', 'customer_age'], inputs={'user_id': 42})
    return time.perf_counter() - t0

async_time = await bench_async()

print(f'Sequential: {sync_time:.3f}s  (0.1 + 0.15 = 0.25s expected)')
print(f'Async:      {async_time:.3f}s  (max(0.1, 0.15) = 0.15s expected)')
print(f'Speedup:    {sync_time / async_time:.1f}×')

### What just happened?
- **Async is ~1.6× faster** for this I/O-bound pipeline — independent nodes ran concurrently.
- For N independent I/O nodes, async scales to `O(max_latency)` instead of `O(sum_of_latencies)`.
- **For pandas/numpy transforms** (CPU-bound, no I/O): async gives no benefit — sync Driver is simpler.

## Step 3 · Parallel Execution — Same Pipeline, Many Inputs

The parallel pattern is different: one pipeline applied to many **independent** inputs (e.g. one row per customer). Hamilton's `Parallelizable` and `Collect` types express fan-out / fan-in.

```python
from hamilton.htypes import Parallelizable, Collect

def customer_ids(raw_ids: list) -> Parallelizable[int]:
    """Fan out: yield one customer_id per parallel branch."""
    for cid in raw_ids:
        yield cid

def score(customer_id: int) -> float:
    """Computed independently per customer_id branch."""
    return customer_id * 0.5

def all_scores(score: Collect[float]) -> list:
    """Fan in: collect all scores from parallel branches."""
    return list(score)
```

In [ ]:
from hamilton.htypes import Parallelizable, Collect
from hamilton.execution import executors

# Fan-out: one branch per customer_id
def customer_id(customer_ids: list) -> Parallelizable[int]:
    """Yield one customer_id per parallel branch."""
    for cid in customer_ids:
        yield cid

# This runs independently on each branch
def customer_score(customer_id: int) -> float:
    """Simulate per-customer scoring — 10ms of work."""
    _time.sleep(0.01)  # simulate work
    return float(customer_id % 100) / 100.0

def customer_tier(customer_score: float) -> str:
    """Tier assignment based on score."""
    if customer_score > 0.75: return 'platinum'
    if customer_score > 0.50: return 'gold'
    if customer_score > 0.25: return 'silver'
    return 'bronze'

# Fan-in: collect all branch results
def all_scores(customer_score: Collect[float]) -> list:
    """Collect scores from all parallel branches."""
    return list(customer_score)

def all_tiers(customer_tier: Collect[str]) -> list:
    """Collect tiers from all parallel branches."""
    return list(customer_tier)

parallel_module = types.ModuleType('parallel_pipeline')
for fn in [customer_id, customer_score, customer_tier, all_scores, all_tiers]:
    setattr(parallel_module, fn.__name__, fn)
sys.modules['parallel_pipeline'] = parallel_module

print('Parallel pipeline defined: fan-out on customer_id, fan-in to all_scores/all_tiers')

In [ ]:
TEST_IDS = list(range(20))  # 20 customers, each takes 10ms

# Sequential execution
dr_seq = (
    driver.Builder()
    .with_modules(parallel_module)
    .build()
)
t0 = time.perf_counter()
seq_result = dr_seq.execute(['all_scores', 'all_tiers'], inputs={'customer_ids': TEST_IDS})
seq_time = time.perf_counter() - t0

# Parallel execution with ThreadPoolExecutor (4 workers)
dr_par = (
    driver.Builder()
    .with_modules(parallel_module)
    .enable_dynamic_execution(allow_experimental_mode=True)
    .build()
)
t0 = time.perf_counter()
par_result = dr_par.execute(
    ['all_scores', 'all_tiers'],
    inputs={'customer_ids': TEST_IDS},
    executor=executors.SynchronousLocalTaskExecutor()  # swap for MultiThreadingExecutor in prod
)
par_time = time.perf_counter() - t0

print(f'Sequential:  {seq_time:.3f}s  (20 × 10ms = ~200ms)')
print(f'Parallel:    {par_time:.3f}s')
print(f'Both produced {len(seq_result["all_scores"])} scores')
print(f'Tier counts: {pd.Series(seq_result["all_tiers"]).value_counts().to_dict()}')

### What just happened?
- **`Parallelizable[T]`** turns a generator into a fan-out — Hamilton creates one branch per yielded value.
- **`Collect[T]`** fans back in — the node receives all branch results as an iterable.
- In production, swap `SynchronousLocalTaskExecutor` for `MultiThreadingExecutor(num_cpus=4)` or a Ray executor for true parallelism.
- The DAG structure is unchanged — only the executor changes to control parallelism.

## Step 4 · Async Pipeline with Multiple Concurrent Requests

In [ ]:
# Run the async pipeline for 5 users concurrently using asyncio.gather
async def run_for_user(uid: int) -> dict:
    return await dr_async.execute(
        ['spend_per_order', 'customer_age', 'is_premium'],
        inputs={'user_id': uid}
    )

async def run_all_users():
    user_ids = [1, 5, 10, 42, 99]
    t0 = time.perf_counter()
    results = await asyncio.gather(*[run_for_user(uid) for uid in user_ids])
    elapsed = time.perf_counter() - t0
    return results, elapsed

results, elapsed = await run_all_users()

# Expected: ~0.15s (all users run concurrently, each takes max(0.1, 0.15)=0.15s)
print(f'5 users processed concurrently in: {elapsed:.3f}s')
print(f'Per-user sequential would take: ~{5 * 0.25:.2f}s')
print(f'\nResults:')
for uid, r in zip([1, 5, 10, 42, 99], results):
    print(f'  user {uid:2d}: spend_per_order={r["spend_per_order"]:.1f}, age={r["customer_age"]}, premium={r["is_premium"]}')

### What just happened?
- **`asyncio.gather`** ran 5 separate pipeline executions concurrently — each with its own `user_id` input.
- Total time ≈ 0.15s instead of 5 × 0.25s = 1.25s — an 8× speedup for this I/O-bound workload.
- **This is the production pattern** for per-entity feature enrichment: one async Driver, N concurrent `execute()` calls via `gather`.

In [ ]:
# Challenge: add a third async I/O node `loyalty_points` that simulates a 200ms
# loyalty database lookup (asyncio.sleep(0.2))
# Then add a node `vip_eligible(is_premium, loyalty_points)` that depends on both
# Measure the new total execution time and verify it's still ~0.2s (max latency)

# async def loyalty_points(user_id: int) -> int:
#     await asyncio.sleep(0.2)
#     return user_id * 100

# async def vip_eligible(is_premium: bool, loyalty_points: int) -> bool:
#     return is_premium and loyalty_points > 3000

print('Add loyalty_points (200ms) and vip_eligible — verify total is ~0.2s, not 0.45s!')

---
## Day 9 key concepts recap

| Concept | What to remember |
|---|---|
| `AsyncDriver` | Same Builder API as `Driver`; `await dr.execute(...)` |
| Concurrent I/O | Independent async nodes run concurrently — total ≈ max latency, not sum |
| Async ≠ faster for CPU | Use sync Driver for pandas/numpy; async only helps with I/O waits |
| `Parallelizable[T]` | Generator that fans out into N independent branches |
| `Collect[T]` | Fans back in — receives all branch results as an iterable |
| `asyncio.gather` | Run the same pipeline for N entities concurrently |

> **Tip:** Async Hamilton is ideal when nodes call external APIs or I/O-bound services. For CPU-bound tasks, use the multiprocessing-backed parallel executor instead.

---
## What's next
**Day 10** → Polars and multiple DataFrame backends: port the Day 4 pipeline to Polars with minimal code changes, and benchmark pandas vs. Polars.

Mark Day 9 complete in your [tracker](../index.html).